# Human in the Loop

This concepts comes when we need human interventions like sending an email, payments, accessing social security numbers. All these require Human eye to approve or reject, for that we require a middleware function for that. 

In [20]:
from dotenv import load_dotenv
load_dotenv()

True

In [21]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]


@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body"""
    # fake email sending
    return f"Email Sent"

In [22]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailState(AgentState):
    email: str


agent = create_agent(
    model="claude-sonnet-4-5",
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True
            },
            description_prefix="Tool execution required approval"
        )
    ]
)

In [23]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Sean, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
    },
    config=config
)

In [14]:
from pprint import pprint

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John,\n'
                                                                          '\n'
                                                                          'No '
                                                                          'problem '
                                                                          'at '
                                                                          'all! '
                                                                          'Yes, '
                                                                          'we '
                                                                          'can '
                                                                          'reschedule. '
                                                                          'What '
                

In [6]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': "Hi John,\n\nNo problem at all! I understand things come up. Let's reschedule - what time works better for you tomorrow, or would you prefer to move it to another day?\n\nBest,\nSean"}, 'description': 'Tool execution required approval\n\nTool: send_email\nArgs: {\'body\': "Hi John,\\n\\nNo problem at all! I understand things come up. Let\'s reschedule - what time works better for you tomorrow, or would you prefer to move it to another day?\\n\\nBest,\\nSean"}'}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='2e76887afd19d33ef449a8c285af0830')]


In [9]:
# Access just the 'body' argument from the tool call

print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John,

No problem at all! I understand things come up. Let's reschedule - what time works better for you tomorrow, or would you prefer to move it to another day?

Best,
Sean


## Approve

In [10]:
from langgraph.types import Command

response = agent.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]}
    ),
    config=config # Same thread ID to resume the paused conversation
)

pprint(response)

{'email': "Hi Sean, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='9e8ea1a6-be5c-4f8b-b596-be6c0c1f2781'),
              AIMessage(content=[{'text': "I'll help you read your email and send a response. Let me start by reading your email.", 'type': 'text'}, {'id': 'toolu_015KQSUTmShUgedKe5E9C42E', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'read_email', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_01G4hCpbpqZdTZYya2m2QZm9', 'container': None, 'model': 'claude-sonnet-4-5-20250929', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_token

## Reject

In [15]:
response = agent.invoke(
    Command(
        resume={"decisions": [
            {
                "type": "reject",
                # An explaination of why the request was rejected
                "message": "No please sign off - Your merciful leader, Seán."
            }
        ]}
    ),
    config=config
)

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John,\n'
                                                                          '\n'
                                                                          'No '
                                                                          'problem '
                                                                          'at '
                                                                          'all! '
                                                                          'Yes, '
                                                                          'we '
                                                                          'can '
                                                                          'reschedule. '
                                                                          'What '
                

## Edit

In [24]:
response = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "This is the last straw, you're fired!"},
                    }
                }
            ]
        }
    ),
    config=config
)

pprint(response)

{'email': "Hi Sean, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='836d3d23-8165-47b1-9d32-ca27c8d36a7a'),
              AIMessage(content=[{'text': "I'll help you read your email and send a response. Let me start by reading your email first.", 'type': 'text'}, {'id': 'toolu_01V7tmYNSZwxFhwPW1vhcD3o', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'read_email', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_015V8YSv5AGfaG7m81HS45ZD', 'container': None, 'model': 'claude-sonnet-4-5-20250929', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input